# REGIME-SHIFT Stage 4 f6 - Optimizer Development

This notebook is intentionally isolated: no HMM walk-forward loop and no strategy backtest.

Stage 4 validates the optimizer surface using locked configuration files, one static covariance estimate, one static scenario window, and explicit CVXPY unit checks for Bull and Crisis regimes.

In [13]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.covariance import LedoitWolf
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.float_format = "{:,.4f}".format

PROJECT_DIR = Path(r"C:\Users\Gangadhar\Documents\Regime_shift_ind")

CONFIG_CANDIDATES = [
    Path.cwd() / "stage4_optimizer_config",
    Path(r"C:\Users\Gangadhar\Documents\Regime Shift Quant\stage4_optimizer_config"),
    PROJECT_DIR / "stage4_optimizer_config",
]
CONFIG_DIR = next((p for p in CONFIG_CANDIDATES if p.exists()), None)
if CONFIG_DIR is None:
    raise FileNotFoundError("stage4_optimizer_config was not found in the notebook cwd, workspace, or project folder.")

OUTPUT_DIR = CONFIG_DIR.parent / "stage4_optimizer_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"cvxpy: {cp.__version__}")
print(f"Installed CVXPY solvers: {cp.installed_solvers()}")
print(f"Project dir: {PROJECT_DIR}")
print(f"Config dir : {CONFIG_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

Imports OK
cvxpy: 1.9.1
Installed CVXPY solvers: ['CLARABEL', 'SCS', 'SCIPY', 'HIGHS', 'OSQP']
Project dir: C:\Users\Gangadhar\Documents\Regime_shift_ind
Config dir : c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_config
Output dir : c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs


## 1. Locked Optimizer Configuration

These files are the Stage 4 source of truth before any backtest is connected:

- `weight_bounds.csv`
- `combined_constraints.csv`
- `optimizer_constants.json`

In [14]:
constants = json.loads((CONFIG_DIR / "optimizer_constants.json").read_text(encoding="utf-8"))
bounds_raw = pd.read_csv(CONFIG_DIR / "weight_bounds.csv")
groups_raw = pd.read_csv(CONFIG_DIR / "combined_constraints.csv")

ASSETS = constants["asset_order"]
REGIMES = constants.get("regimes", ["Bull", "Crisis"])
N_ASSETS = len(ASSETS)
CVaR_CONFIDENCE = float(constants["cvar_confidence"])
EXPECTED_RETURN_METHOD = constants["expected_return_method"]
SOLVER_ORDER = constants["solver_order"]
OBJECTIVE_WEIGHTS = constants["objective_weights"]

missing_bounds = sorted(set(REGIMES) - set(bounds_raw["regime"]))
if missing_bounds:
    raise ValueError(f"Missing bounds for regimes: {missing_bounds}")

missing_assets = sorted(set(ASSETS) - set(bounds_raw["asset"]))
if missing_assets:
    raise ValueError(f"Missing bounds for assets: {missing_assets}")

bounds_locked = bounds_raw.copy()

cost_bps = pd.DataFrame(constants["transaction_cost_bps"]).T.loc[ASSETS, ["buy", "sell"]].astype(float)
BUY_COSTS = cost_bps["buy"].to_numpy() / 10_000
SELL_COSTS = cost_bps["sell"].to_numpy() / 10_000

print("Locked Stage 4 configuration loaded")
print(f"CVaR confidence level     : {CVaR_CONFIDENCE:.0%}")
print(f"Expected return method    : {EXPECTED_RETURN_METHOD}")
print("Transaction costs in bps:")
display(cost_bps)

print("Locked regime bounds:")
display(bounds_locked.pivot(index="asset", columns="regime", values=["lower", "upper"]).loc[ASSETS])

print("Combined-category concentration constraints:")
display(groups_raw)

Locked Stage 4 configuration loaded
CVaR confidence level     : 95%
Expected return method    : minimum_variance
Transaction costs in bps:


,buy,sell
NIFTYBEES,3.5000,13.0000
JUNIORBEES,3.5000,13.0000
GOLDBEES,2.5000,2.5000
LIQUIDBEES,1.0000,1.0000


Locked regime bounds:


lower                upper              
regime       Bear   Bull Crisis   Bear   Bull Crisis
asset                                               
NIFTYBEES  0.1500 0.4000 0.0000 0.3500 0.6500 0.1000
JUNIORBEES 0.0000 0.1500 0.0000 0.1500 0.3500 0.0500
GOLDBEES   0.2000 0.0000 0.2500 0.4500 0.1500 0.5000
LIQUIDBEES 0.1000 0.0000 0.2500 0.3500 0.1000 0.5000

Combined-category concentration constraints:


,regime,group,assets,lower,upper,note
0,Bull,Equity,NIFTYBEES|JUNIORBEES,0.6000,0.9000,Keep bull portfolios structurally risk-on with...
1,Bull,Defensive,GOLDBEES|LIQUIDBEES,0.1000,0.4000,Maintain a modest ballast sleeve
2,Bull,SafeHaven,GOLDBEES,0.0500,0.3000,Prevent full defensive crowding in bull regimes
3,Bull,Cash,LIQUIDBEES,0.0000,0.1000,Limit cash drag
4,Bear,Equity,NIFTYBEES|JUNIORBEES,0.1500,0.5000,Partial de-risking while keeping upside partic...
5,Bear,Defensive,GOLDBEES|LIQUIDBEES,0.5000,0.8500,Require majority defensive allocation
6,Bear,SafeHaven,GOLDBEES,0.3000,0.7000,Limit single-category concentration while reta...
7,Bear,Cash,LIQUIDBEES,0.1000,0.3500,Require liquidity buffer
8,Crisis,Equity,NIFTYBEES|JUNIORBEES,0.0000,0.1500,Hard cap on equity drawdown exposure
9,Crisis,Defensive,GOLDBEES|LIQUIDBEES,0.8500,1.0000,Force defensive capital preservation posture


## 2. Static Data Window

The optimizer is validated on a single monthly estimation window using only the four current project assets: `NIFTYBEES`, `JUNIORBEES`, `GOLDBEES`, and `LIQUIDBEES`.

In [15]:
def load_price_series(asset: str) -> pd.Series:
    path = PROJECT_DIR / f"{asset}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    if asset in df.columns:
        series = df[asset]
    else:
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) == 0:
            raise ValueError(f"No numeric price column found in {path}")
        series = df[numeric_cols[0]]
    series = series.copy()
    series.index = pd.to_datetime(series.index)
    series = series.sort_index().dropna()
    return series.rename(asset)


prices = pd.concat([load_price_series(asset) for asset in ASSETS], axis=1).dropna(how="any")
monthly_prices = prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change().dropna(how="any")

window_start = pd.Timestamp(constants["static_window_start"])
window_end = pd.Timestamp(constants["static_window_end"])
scenario_returns = monthly_returns.loc[window_start:window_end, ASSETS].dropna()
if len(scenario_returns) < 36:
    raise ValueError(f"Static scenario window too short: {len(scenario_returns)} rows")

lw = LedoitWolf().fit(scenario_returns.to_numpy())
static_cov = pd.DataFrame(lw.covariance_, index=ASSETS, columns=ASSETS)

annual_vol = pd.Series(
    {
        "NIFTYBEES": 0.18,
        "JUNIORBEES": 0.24,
        "GOLDBEES": 0.14,
        "LIQUIDBEES": 0.01,
    }
).loc[ASSETS]
monthly_vol = annual_vol / np.sqrt(12)
known_corr = pd.DataFrame(
    [
        [1.00, 0.85, -0.05, 0.05],
        [0.85, 1.00, -0.10, 0.05],
        [-0.05, -0.10, 1.00, 0.10],
        [0.05, 0.05, 0.10, 1.00],
    ],
    index=ASSETS,
    columns=ASSETS,
)
known_cov = pd.DataFrame(
    np.outer(monthly_vol, monthly_vol) * known_corr.to_numpy(),
    index=ASSETS,
    columns=ASSETS,
)

def nearest_psd(matrix: np.ndarray, floor: float = 1e-10) -> np.ndarray:
    sym = (np.asarray(matrix, dtype=float) + np.asarray(matrix, dtype=float).T) / 2
    eigvals, eigvecs = np.linalg.eigh(sym)
    eigvals = np.maximum(eigvals, floor)
    psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    return (psd + psd.T) / 2


# Correct the known test matrix only if numerical roundoff makes it non-PSD.
if np.linalg.eigvalsh(known_cov.to_numpy()).min() < -1e-10:
    known_cov = pd.DataFrame(nearest_psd(known_cov.to_numpy()), index=ASSETS, columns=ASSETS)

print("Static monthly data ready")
print(f"Price panel      : {prices.index.min().date()} -> {prices.index.max().date()}, shape={prices.shape}")
print(f"Scenario window  : {scenario_returns.index.min().date()} -> {scenario_returns.index.max().date()}, rows={len(scenario_returns)}")
print(f"Expected returns : {EXPECTED_RETURN_METHOD}; mu vector is not used in objectives")

summary = pd.DataFrame(
    {
        "ann_return": (1 + scenario_returns).prod() ** (12 / len(scenario_returns)) - 1,
        "ann_vol": scenario_returns.std() * np.sqrt(12),
        "min_month": scenario_returns.min(),
        "max_month": scenario_returns.max(),
    }
)
display((summary * 100).round(2))

Static monthly data ready
Price panel      : 2009-05-31 -> 2024-12-29, shape=(3844, 4)
Scenario window  : 2019-01-31 -> 2023-12-31, rows=60
Expected returns : minimum_variance; mu vector is not used in objectives


,ann_return,ann_vol,min_month,max_month
NIFTYBEES,16.0900,19.3800,-25.1000,16.6000
JUNIORBEES,14.4200,19.6000,-20.9800,14.5400
GOLDBEES,13.7800,13.2200,-5.6200,11.3200
LIQUIDBEES,4.0400,0.5400,0.0100,0.7600


## 3. CVXPY Optimizer Helpers

The solver does four defensive checks:

1. bounds feasibility before optimization,
2. combined-category feasibility,
3. CVXPY status check,
4. bounded equal-weight fallback on infeasible, failed, or singular covariance cases.

In [16]:
GOOD_STATUSES = {cp.OPTIMAL, cp.OPTIMAL_INACCURATE}
TOL = 1e-6


def regime_bounds(regime: str):
    sub = bounds_locked[bounds_locked["regime"].eq(regime)].set_index("asset").loc[ASSETS]
    lower = sub["lower"].astype(float).to_numpy()
    upper = sub["upper"].astype(float).to_numpy()
    return lower, upper


def regime_group_rows(regime: str):
    rows = []
    for row in groups_raw[groups_raw["regime"].eq(regime)].to_dict("records"):
        assets = [asset for asset in row["assets"].split("|") if asset in ASSETS]
        rows.append(
            {
                "group": row["group"],
                "assets": assets,
                "idx": [ASSETS.index(asset) for asset in assets],
                "lower": float(row["lower"]),
                "upper": float(row["upper"]),
            }
        )
    return rows


def add_weight_constraints(w, regime: str):
    lower, upper = regime_bounds(regime)
    constraints = [cp.sum(w) == 1, w >= lower, w <= upper]
    for row in regime_group_rows(regime):
        group_weight = cp.sum(w[row["idx"]])
        constraints.append(group_weight >= row["lower"])
        constraints.append(group_weight <= row["upper"])
    return constraints


def solve_cvxpy_problem(problem):
    installed = set(cp.installed_solvers())
    last_error = None
    for solver in SOLVER_ORDER:
        if solver not in installed:
            continue
        try:
            value = problem.solve(solver=solver, verbose=False)
            return problem.status, solver, value, None
        except Exception as exc:
            last_error = repr(exc)
    return "solver_exception", None, np.nan, last_error


def feasibility_check(regime: str):
    lower, upper = regime_bounds(regime)
    if lower.sum() > 1 + TOL:
        return False, "lower_bounds_sum_above_one"
    if upper.sum() < 1 - TOL:
        return False, "upper_bounds_sum_below_one"
    if np.any(lower > upper + TOL):
        return False, "lower_bound_above_upper_bound"

    w = cp.Variable(N_ASSETS)
    problem = cp.Problem(cp.Minimize(cp.sum_squares(w - np.ones(N_ASSETS) / N_ASSETS)), add_weight_constraints(w, regime))
    status, solver, _, error = solve_cvxpy_problem(problem)
    if status not in GOOD_STATUSES:
        return False, f"cvxpy_feasibility_failed:{status}:{error}"
    return True, f"feasible:{solver}"


def clean_weights(raw_weights: np.ndarray):
    weights = np.asarray(raw_weights, dtype=float).ravel()
    weights[np.abs(weights) < 1e-9] = 0
    weights = np.clip(weights, 0, 1)
    total = weights.sum()
    if total <= 0:
        return np.ones(N_ASSETS) / N_ASSETS
    return weights / total


def fallback_equal_weight(regime: str, reason: str):
    w = cp.Variable(N_ASSETS)
    target = np.ones(N_ASSETS) / N_ASSETS
    problem = cp.Problem(cp.Minimize(cp.sum_squares(w - target)), add_weight_constraints(w, regime))
    status, solver, _, error = solve_cvxpy_problem(problem)
    if status in GOOD_STATUSES and w.value is not None:
        weights = clean_weights(w.value)
        return {
            "regime": regime,
            "weights": weights,
            "status": f"fallback_{reason}",
            "solver": solver,
            "fallback": True,
            "objective_value": np.nan,
            "error": error,
        }

    lower, upper = regime_bounds(regime)
    weights = lower.copy()
    remaining = 1 - weights.sum()
    capacity = np.maximum(upper - lower, 0)
    if remaining > 0 and capacity.sum() > 0:
        weights += remaining * capacity / capacity.sum()
    return {
        "regime": regime,
        "weights": clean_weights(weights),
        "status": f"fallback_{reason}_projection",
        "solver": None,
        "fallback": True,
        "objective_value": np.nan,
        "error": error,
    }


def covariance_health(sigma: np.ndarray):
    sym = (np.asarray(sigma, dtype=float) + np.asarray(sigma, dtype=float).T) / 2
    eigvals = np.linalg.eigvalsh(sym)
    return {
        "min_eig": float(eigvals.min()),
        "rank": int(np.linalg.matrix_rank(sym, tol=1e-10)),
        "is_psd": bool(eigvals.min() >= -1e-10),
    }


def transaction_cost_expr(w, previous_weights):
    previous_weights = np.asarray(previous_weights, dtype=float)
    delta = w - previous_weights
    buy_cost = cp.sum(cp.multiply(BUY_COSTS, cp.pos(delta)))
    sell_cost = cp.sum(cp.multiply(SELL_COSTS, cp.pos(-delta)))
    return buy_cost + sell_cost


def empirical_cvar(losses: np.ndarray, alpha: float = CVaR_CONFIDENCE):
    losses = np.asarray(losses, dtype=float)
    cutoff = np.quantile(losses, alpha)
    tail = losses[losses >= cutoff]
    return float(tail.mean()) if len(tail) else float(cutoff)


def solve_regime_objective(regime: str, sigma, scenarios, previous_weights=None, reject_singular=False):
    feasible, reason = feasibility_check(regime)
    if not feasible:
        return fallback_equal_weight(regime, reason)

    sigma = np.asarray(sigma, dtype=float)
    health = covariance_health(sigma)
    if not health["is_psd"]:
        sigma = nearest_psd(sigma)
        health = covariance_health(sigma)

    if reject_singular and health["rank"] < N_ASSETS:
        return fallback_equal_weight(regime, "singular_covariance")

    previous_weights = np.ones(N_ASSETS) / N_ASSETS if previous_weights is None else np.asarray(previous_weights, dtype=float)
    scenarios = np.asarray(scenarios, dtype=float)

    w = cp.Variable(N_ASSETS)
    eta = cp.Variable()
    u = cp.Variable(len(scenarios))
    losses = -scenarios @ w
    cvar_expr = eta + cp.sum(u) / ((1 - CVaR_CONFIDENCE) * len(scenarios))

    constraints = add_weight_constraints(w, regime)
    constraints.extend([u >= 0, u >= losses - eta])

    settings = OBJECTIVE_WEIGHTS[regime]
    variance_expr = cp.quad_form(w, cp.psd_wrap(sigma))
    tc_expr = transaction_cost_expr(w, previous_weights)
    objective = (
        settings["variance"] * variance_expr
        + settings["cvar"] * cvar_expr
        + settings["transaction_cost"] * tc_expr
    )

    problem = cp.Problem(cp.Minimize(objective), constraints)
    status, solver, value, error = solve_cvxpy_problem(problem)
    if status not in GOOD_STATUSES or w.value is None:
        return fallback_equal_weight(regime, f"cvxpy_status_{status}")

    return {
        "regime": regime,
        "weights": clean_weights(w.value),
        "status": status,
        "solver": solver,
        "fallback": False,
        "objective_value": float(value),
        "error": error,
    }


def validate_weights(regime: str, weights):
    weights = np.asarray(weights, dtype=float)
    lower, upper = regime_bounds(regime)
    group_values = {}
    group_ok = True
    for row in regime_group_rows(regime):
        value = float(weights[row["idx"]].sum())
        group_values[row["group"]] = value
        group_ok = group_ok and (value >= row["lower"] - 5e-5) and (value <= row["upper"] + 5e-5)
    return {
        "sum": float(weights.sum()),
        "sum_ok": bool(abs(weights.sum() - 1) <= 5e-5),
        "bounds_ok": bool(np.all(weights >= lower - 5e-5) and np.all(weights <= upper + 5e-5)),
        "groups_ok": bool(group_ok),
        **{f"group_{key}": val for key, val in group_values.items()},
    }


def portfolio_metrics(weights, sigma, scenarios, previous_weights=None):
    weights = np.asarray(weights, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    scenarios = np.asarray(scenarios, dtype=float)
    previous_weights = np.ones(N_ASSETS) / N_ASSETS if previous_weights is None else np.asarray(previous_weights, dtype=float)
    losses = -(scenarios @ weights)
    delta = weights - previous_weights
    return {
        "ann_vol": float(np.sqrt(max(weights @ sigma @ weights, 0)) * np.sqrt(12)),
        "monthly_cvar_loss": empirical_cvar(losses),
        "one_way_turnover": float(np.abs(delta).sum() / 2),
        "buy_cost_bps": float(np.maximum(delta, 0) @ (BUY_COSTS * 10_000)),
        "sell_cost_bps": float(np.maximum(-delta, 0) @ (SELL_COSTS * 10_000)),
    }


print("Optimizer helper functions ready")

Optimizer helper functions ready


## 4. Pre-solve Feasibility Checks

Every regime must be feasible before optimization is attempted.

In [17]:
feasibility_rows = []
for regime in REGIMES:
    feasible, reason = feasibility_check(regime)
    lower, upper = regime_bounds(regime)
    feasibility_rows.append(
        {
            "regime": regime,
            "feasible": feasible,
            "reason": reason,
            "lower_sum": lower.sum(),
            "upper_sum": upper.sum(),
        }
    )

feasibility_df = pd.DataFrame(feasibility_rows)
display(feasibility_df)
assert feasibility_df["feasible"].all(), "At least one regime constraint set is infeasible."
print("Pre-solve feasibility checks passed for Bull and Crisis.")

,regime,feasible,reason,lower_sum,upper_sum
0,Bull,True,feasible:CLARABEL,0.5500,1.2500
1,Crisis,True,feasible:CLARABEL,0.5000,1.1500


Pre-solve feasibility checks passed for Bull and Crisis.


## 5. Isolated Objective Tests With Known Covariance

These are unit-style checks against a controlled covariance matrix. The objective functions are still CVXPY problems, but no HMM probabilities or walk-forward refits are involved.

In [18]:
known_scenarios = scenario_returns[ASSETS].to_numpy()
previous_weights = np.ones(N_ASSETS) / N_ASSETS

unit_solutions = {}
unit_rows = []
unit_weight_rows = []

for regime in REGIMES:
    solution = solve_regime_objective(
        regime=regime,
        sigma=known_cov.to_numpy(),
        scenarios=known_scenarios,
        previous_weights=previous_weights,
        reject_singular=False,
    )
    weights = solution["weights"]
    validation = validate_weights(regime, weights)
    metrics = portfolio_metrics(weights, known_cov.to_numpy(), known_scenarios, previous_weights)
    unit_solutions[regime] = solution

    row = {
        "regime": regime,
        "status": solution["status"],
        "solver": solution["solver"],
        "fallback": solution["fallback"],
        **validation,
        **metrics,
    }
    unit_rows.append(row)
    unit_weight_rows.append(pd.Series(weights, index=ASSETS, name=regime))

unit_weights = pd.DataFrame(unit_weight_rows)
unit_results = pd.DataFrame(unit_rows)

print("Known-covariance optimized weights:")
display((unit_weights * 100).round(2))

print("Known-covariance validation metrics:")
display(unit_results)

assert not unit_results["fallback"].any(), "Known covariance tests should solve without fallback."
assert unit_results["sum_ok"].all(), "Weights do not sum to one."
assert unit_results["bounds_ok"].all(), "At least one solution violates asset bounds."
assert unit_results["groups_ok"].all(), "At least one solution violates combined-category constraints."
assert unit_results.loc[unit_results["regime"].eq("Bull"), "group_Equity"].iloc[0] >= 0.75 - 1e-6
assert unit_results.loc[unit_results["regime"].eq("Crisis"), "group_Defensive"].iloc[0] >= 0.85 - 1e-6

print("Bull and Crisis objective tests passed.")

Known-covariance optimized weights:


,NIFTYBEES,JUNIORBEES,GOLDBEES,LIQUIDBEES
Bull,50.0000,25.0000,15.0000,10.0000
Crisis,10.0000,0.0000,45.0000,45.0000


Known-covariance validation metrics:


,regime,status,solver,fallback,sum,sum_ok,bounds_ok,groups_ok,group_Equity,group_Defensive,group_SafeHaven,group_Cash,ann_vol,monthly_cvar_loss,one_way_turnover,buy_cost_bps,sell_cost_bps
0,Bull,optimal,CLARABEL,False,1.0000,True,True,True,0.7500,0.2500,0.1500,0.1000,0.1446,0.0872,0.2500,0.8750,0.4000
1,Crisis,optimal,CLARABEL,False,1.0000,True,True,True,0.1000,0.9000,0.4500,0.4500,0.0653,0.0176,0.4000,0.7000,5.2000


Bull and Crisis objective tests passed.


## 6. Singular Covariance Failure Handling

Stage 4 treats singular covariance matrices as production failures and sends them to bounded equal-weight fallback. This avoids unstable optimizer behavior before walk-forward integration.

In [19]:
singular_cov = np.ones((N_ASSETS, N_ASSETS)) * 0.0001
singular_health = covariance_health(singular_cov)
singular_solution = solve_regime_objective(
    regime="Bull",
    sigma=singular_cov,
    scenarios=known_scenarios,
    previous_weights=previous_weights,
    reject_singular=True,
)
singular_weights = pd.Series(singular_solution["weights"], index=ASSETS, name="Bull singular fallback")
singular_validation = validate_weights("Bull", singular_solution["weights"])

print("Singular covariance health:", singular_health)
print("Singular covariance test status:", singular_solution["status"])
display((singular_weights * 100).round(2).to_frame("weight_pct"))
display(pd.DataFrame([singular_validation]))

assert singular_solution["fallback"], "Singular covariance should activate fallback."
assert singular_solution["status"] == "fallback_singular_covariance"
assert singular_validation["sum_ok"] and singular_validation["bounds_ok"] and singular_validation["groups_ok"]
print("Degenerate covariance fallback test passed.")

Singular covariance health: {'min_eig': 0.0, 'rank': 1, 'is_psd': True}
Singular covariance test status: fallback_singular_covariance


,weight_pct
NIFTYBEES,40.0000
JUNIORBEES,35.0000
GOLDBEES,15.0000
LIQUIDBEES,10.0000


,sum,sum_ok,bounds_ok,groups_ok,group_Equity,group_Defensive,group_SafeHaven,group_Cash
0,1.0000,True,True,True,0.7500,0.2500,0.1500,0.1000


Degenerate covariance fallback test passed.


## 7. Static Historical Covariance Sanity Check

This repeats the objective tests using the single Ledoit-Wolf covariance estimate from the static data window. It is still not a backtest.

In [20]:
static_rows = []
static_weight_rows = []

for regime in REGIMES:
    solution = solve_regime_objective(
        regime=regime,
        sigma=static_cov.to_numpy(),
        scenarios=scenario_returns[ASSETS].to_numpy(),
        previous_weights=previous_weights,
        reject_singular=True,
    )
    weights = solution["weights"]
    validation = validate_weights(regime, weights)
    metrics = portfolio_metrics(weights, static_cov.to_numpy(), scenario_returns[ASSETS].to_numpy(), previous_weights)
    static_rows.append(
        {
            "regime": regime,
            "status": solution["status"],
            "solver": solution["solver"],
            "fallback": solution["fallback"],
            **validation,
            **metrics,
        }
    )
    static_weight_rows.append(pd.Series(weights, index=ASSETS, name=regime))

static_weights = pd.DataFrame(static_weight_rows)
static_results = pd.DataFrame(static_rows)

print("Static historical covariance weights:")
display((static_weights * 100).round(2))

print("Static historical covariance validation metrics:")
display(static_results)

assert static_results["sum_ok"].all()
assert static_results["bounds_ok"].all()
assert static_results["groups_ok"].all()
print("Static historical covariance sanity checks passed.")

Static historical covariance weights:


,NIFTYBEES,JUNIORBEES,GOLDBEES,LIQUIDBEES
Bull,50.0000,25.0000,15.0000,10.0000
Crisis,10.0000,0.0000,45.0000,45.0000


Static historical covariance validation metrics:


,regime,status,solver,fallback,sum,sum_ok,bounds_ok,groups_ok,group_Equity,group_Defensive,group_SafeHaven,group_Cash,ann_vol,monthly_cvar_loss,one_way_turnover,buy_cost_bps,sell_cost_bps
0,Bull,optimal,CLARABEL,False,1.0000,True,True,True,0.7500,0.2500,0.1500,0.1000,0.1330,0.0872,0.2500,0.8750,0.4000
1,Crisis,optimal,CLARABEL,False,1.0000,True,True,True,0.1000,0.9000,0.4500,0.4500,0.0709,0.0176,0.4000,0.7000,5.2000


Static historical covariance sanity checks passed.


## 8. Save Stage 4 Optimizer Development Artifacts

The files below are optimizer-development artifacts only. They are not walk-forward backtest outputs.

In [21]:
bounds_locked.to_csv(OUTPUT_DIR / "locked_weight_bounds.csv", index=False)
groups_raw.to_csv(OUTPUT_DIR / "locked_combined_constraints.csv", index=False)
unit_weights.to_csv(OUTPUT_DIR / "known_covariance_unit_weights.csv")
unit_results.to_csv(OUTPUT_DIR / "known_covariance_unit_results.csv", index=False)
static_weights.to_csv(OUTPUT_DIR / "static_covariance_weights.csv")
static_results.to_csv(OUTPUT_DIR / "static_covariance_results.csv", index=False)

metadata = {
    "stage": "Stage 4 isolated optimizer development",
    "no_walk_forward": True,
    "assets": ASSETS,
    "cvar_confidence": CVaR_CONFIDENCE,
    "expected_return_method": EXPECTED_RETURN_METHOD,
    "static_window_start": str(scenario_returns.index.min().date()),
    "static_window_end": str(scenario_returns.index.max().date()),
    "singular_covariance_status": singular_solution["status"],
    "known_covariance_statuses": unit_results[["regime", "status", "fallback"]].to_dict("records"),
    "static_covariance_statuses": static_results[["regime", "status", "fallback"]].to_dict("records"),
}
(OUTPUT_DIR / "stage4_optimizer_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved Stage 4 isolated optimizer artifacts:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {path}  ({path.stat().st_size / 1024:.1f} KB)")

print("Stage 4 isolated optimizer development complete.")

Saved Stage 4 isolated optimizer artifacts:
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\known_covariance_unit_results.csv  (0.6 KB)
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\known_covariance_unit_weights.csv  (0.2 KB)
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\locked_combined_constraints.csv  (1.0 KB)
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\locked_weight_bounds.csv  (0.7 KB)
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\stage4_optimizer_metadata.json  (0.8 KB)
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\static_covariance_results.csv  (0.6 KB)
  c:\Users\Gangadhar\Documents\Regime_shift_ind\stage4_optimizer_outputs\static_covariance_weights.csv  (0.2 KB)
Stage 4 isolated optimizer development complete.
